# Hotelling Model + Stock Pollution Model (Additional Self Study)
**Exercise Session Resource Economics (Spring Term 2026)** \
Raul Hochuli (raul.hochuli@unibas.ch)\ 

This notebook is an extension of `2_Exc_Hotelling_StockPoll_ResEcon26.ipynb` to show how the model reacts on parameter changes and how it can be adjusted to different circumstances. 
We will spend a few minutes in the next exercise session to convey the most important intuition of these model adjustments, but not examine the entire code in detail

In [ ]:
# Packages used in the notebook
import numpy as np
import pandas as pd
import scipy.optimize as opt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1 Hotelling Model - Additional Exercises

Consider again the model we setup in the exercises session to solve the following additional exercises. 

In [ ]:
def hotelling_mono(T = 50, S0 = 100, a = 10, b = 1, c = 1, r = 0.05):
    # Sets & Parameters --------------------------------
    rho = 1/(1+r)**np.arange(T+1)

    # State Variables -----------------------------------
    def f_p(i_q):
        return a - b*i_q

    def f_S(i_q):
        S = np.zeros(T+1)
        S[0] = S0
        for t in range(1, T+1):
            S[t] = S[t-1] - i_q[t-1]
        return S

    # Objective Function -------------------------------
    def f_npv(i_q):
        npv = rho * (i_q * (f_p(i_q) - c))
        return npv
        
    def f_obj(i_q):
        obj_value = sum( f_npv(i_q) )
        return -1* obj_value


    # Decision Variables -------------------------------
    q_start = np.full(T+1, 1)
    bnds = [(0, S0) for t in range(T+1)]

    # Constraints --------------------------------------
    def c_total_extr(i_q):
        return S0 - sum([i_q[t] for t in range(T+1)])
    cnstr = [
        {'type': 'eq', 'fun': c_total_extr},
    ]

    # Run Optimization ---------------------------------
    results = opt.minimize(f_obj, q_start, bounds=bnds, constraints=cnstr,
                        options={'disp': False}
                        )

    # Results ------------------------------------------
    q_opt = results.x
    S_opt = f_S(q_opt)
    p_opt = f_p(q_opt)
    npv_opt = f_npv(q_opt) 

    df = pd.DataFrame({
        't': range(T+1),
        'q_opt': q_opt,
        'S_opt': S_opt,
        'p_opt': p_opt,
        'npv_opt': npv_opt
    })

    return df

In [ ]:
# this plotting function is a small helper function, to visualize results quickly without much additional code afterwards 
def add_trace_to_subplot_hotelling(fig, df, legend_name):
    fig.add_trace(go.Scatter(x=df['t'], y=df['q_opt'], mode='lines', name=f'q_{legend_name}', legendgroup=legend_name, showlegend=True), row=1, col=1)
    fig.add_trace(go.Scatter(x=df['t'], y=df['p_opt'], mode='lines', name=f'p_{legend_name}', legendgroup=legend_name, showlegend=True), row=1, col=2)
    fig.add_trace(go.Scatter(x=df['t'], y=df['S_opt'], mode='lines', name=f'S_{legend_name}', legendgroup=legend_name, showlegend=True), row=1, col=3)
    
    return fig

### 1a) Parameter Effects

To get a better understanding of the model and to verify the suggested effects discussed during the lecture, you can rerun the model with different specifications to see how different parameter changes alter the optimal extraction schedule an price path of the resource.

- Increase interest rate: $r^* = 0.07$
- Increase resource stock: $S_0^* = 150$
- Change in the inverse demand function with $b^* = 0.5$
- Increase in extraction cost with $c^* = 2$
 

In [ ]:
df_hotmono_base         = hotelling_mono()
# df_hotmono_incrint      = hotelling_mono() # (...   )

# Plot Results -------------------------------------
fig_compare = make_subplots(rows=1, cols=3, subplot_titles=('q_opt', 'p_opt', 'S_opt'))

fig_compare = add_trace_to_subplot_hotelling(fig_compare, df_hotmono_base,        'base')
# ...

fig_compare.update_layout(title='Hotelling_mono, compare parameter changes', xaxis_title='t', yaxis_title='value', template='plotly_white', width = 1200, height = 500)
fig_compare.show()

### 1b) Extraction Ban Effect

Consider again the case of the monopolist. We now want to take a first step into modeling policy making. Assume the local government imposes legislation that bans all extraction after $T^* = 30$ years. What effect would this have on the extraction schedule given the Hotelling-rule?

In [ ]:
def hotelling_mono_extrban_(T=50, S0=100, a=10, b=1, c=1, r=0.05, t_extr_ban=None):
    # Sets & Parameters --------------------------------
    rho = 1 / (1 + r) ** np.arange(T + 1)
    
    # State Variables -----------------------------------
    def f_S(i_q):
        S = np.zeros(T + 1)
        S[0] = S0
        for t in range(1, T + 1):
            S[t] = S[t - 1] - i_q[t - 1]
        return S

    def f_p(i_q):
        return a - b * i_q

    # Objective Function -------------------------------
    def f_npv(i_q):
        npv = rho * (i_q * (f_p(i_q) - c))
        return npv
        
    def f_obj(i_q):
        obj_value = sum( f_npv(i_q) )
        return -1* obj_value


    # Decision Variables -------------------------------
    q_start = np.full(T + 1, 1)
    # bnds = [(0, S0) for t in range(T + 1)]
    # bnds_befr_ban = # ...
    # bnds_aftr_ban = # ...
    bnds = []

    # Constraints --------------------------------------
    def c_total_extr(i_q):
        return S0 - np.sum(i_q)

    cnstr = [{'type': 'eq', 'fun': c_total_extr}]
    
    # Run Optimization ---------------------------------
    results = opt.minimize(f_obj, q_start, bounds=bnds, constraints=cnstr, 
                           options={'disp': False})

    # Results ------------------------------------------
    q_opt = results.x
    S_opt = f_S(q_opt)
    p_opt = f_p(q_opt)
    npv_opt = [rho[t] * (q_opt[t] * (a - b * q_opt[t]) - c * q_opt[t]) for t in range(T + 1)]

    df = pd.DataFrame({
        't': range(T + 1),
        'q_opt': q_opt,
        'S_opt': S_opt,
        'p_opt': p_opt,
        'npv_opt': npv_opt
    })

    return df



In [ ]:

df_hotelling_mono = hotelling_mono()
df_hotelling_mono_ban = hotelling_mono_extrban(t_extr_ban=30)

# Check results
df_print = df_hotelling_mono.copy()
print('\nMonopolist Results:')
print(f'q_opt[0]:\t{round(df_print["q_opt"][0], 3)} \np_opt[0]:\t{round(df_print["p_opt"][0], 3)} \nsum(q_opt):\t{round(df_print["q_opt"].sum(), 3)} \nsum(npv_opt):\t{round(df_print["npv_opt"].sum(), 3)}')
df_print = df_hotelling_mono_ban.copy()
print('\nMonopolist Results w Extraction Ban:')
print(f'q_opt[0]:\t{round(df_print["q_opt"][0], 3)} \np_opt[0]:\t{round(df_print["p_opt"][0], 3)} \nsum(q_opt):\t{round(df_print["q_opt"].sum(), 3)} \nsum(npv_opt):\t{round(df_print["npv_opt"].sum(), 3)}')


# Visualize results
fig = make_subplots(rows=1, cols=3, subplot_titles=('q_opt', 'p_opt', 'S_opt'))
add_trace_to_subplot_hotelling(fig, df_hotelling_mono, 'mono_base')
add_trace_to_subplot_hotelling(fig, df_hotelling_mono_ban, 'mono_extrban')

fig.update_layout(title = 'Hotelling model base vs ban at T*', xaxis_title='t', yaxis_title='value', template='plotly_white', width = 1200, height = 500)
fig.show()


***Explanaiton for solution***: \
Because the monopolist knows, she cannot fully extract the resource over an optimal time, she increases the extraction rate (causing lower prices) to get as much profit during the shortend time frame as possible. This undesired effect is also know as the "green paradox" or the "rush to burn"

### 1c) Different Solver Engines

As mentioned above there is a [large number of solvers](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize) that you can apply to to `scipy.optimize.minimize()`. 

Loop through `hotelling_compmarket_adj()` with different solvers to illustrate how outcomes can change. 


In [ ]:
def hotelling_mono_solvers(T = 50, S0 = 100, a = 10, b = 1, c = 1, r = 0.05, solver_method = 'SLSQP'):
    # Sets & Parameters --------------------------------
    rho = 1/(1+r)**np.arange(T+1)

    # State Variables -----------------------------------

    def f_p(i_q):
        return a - b*i_q

    def f_S(i_q):
        S = np.zeros(T+1)
        S[0] = S0
        for t in range(1, T+1):
            S[t] = S[t-1] - i_q[t-1]
        return S

    # Objective Function -------------------------------
    def f_npv(i_q):
        npv = rho * (i_q * (f_p(i_q) - c))
        return npv
        
    def f_obj(i_q):
        obj_value = sum( f_npv(i_q) )
        return -1* obj_value


    # Decision Variables -------------------------------
    q_start = np.full(T+1, 1)
    bnds = [(0, S0) for t in range(T+1)]

    # Constraints --------------------------------------
    def c_total_extr(i_q):
        return S0 - sum([i_q[t] for t in range(T+1)])
    cnstr = [
        {'type': 'eq', 'fun': c_total_extr},
    ]

    # Run Optimization ---------------------------------
    results = opt.minimize(f_obj, q_start, bounds=bnds, constraints=cnstr,
                           # ...
                            options={'disp': False}
                        )

    # Results ------------------------------------------
    q_opt = results.x
    S_opt = f_S(q_opt)
    p_opt = f_p(q_opt)
    npv_opt = f_npv(q_opt) # also possible to calculcate this array by hand: [rho[t] * (q_opt[t] * (a-b*q_opt[t]) - c*q_opt[t]) for t in range(T+1)]

    df = pd.DataFrame({
        't': range(T+1),
        'q_opt': q_opt,
        'S_opt': S_opt,
        'p_opt': p_opt,
        'npv_opt': npv_opt
    })

    return df


In [ ]:
fig_mono_compmarket = make_subplots(rows=1, cols=3, subplot_titles=('q_opt', 'p_opt', 'S_opt'))

solver_method_list = [
    'SLSQP',
    # ...
    ]

for solver_method in solver_method_list:
    try: 
        # ...
        add_trace_to_subplot_hotelling(fig_mono_compmarket, df, solver_method)  
    except Exception as e:
        print(f"Solver {solver_method} failed with error: {e}")

fig_mono_compmarket.update_layout(title='Hotelling_compmarket_adj, different solvers', xaxis_title='t', yaxis_title='value', template='plotly_white', width = 1200, height = 500)
fig_mono_compmarket.show()



## 2 Stock Pollution - Additional Exercises

Consider again the model we setup in the exercises session to solve the following additional exercises. 

$$
    W = \sum_{t=0}^{\infty} \rho^t( U(e_t) - D(S_t)) 
    \\[2em]
    S_{t+1} = S_t - f(S_t) + e_t 
    \\[1em]
    f(S_t) = aS_t
    \\[2em]
    U(e_t) = e^b_t
    \\[1em]
    D(S_t) = cS_t^d
    \\[2em]
    \rho_t = \frac{1}{(1+r)^t}
$$



In [ ]:
def stockpollution(T = 50, S0 = 0, a = 0.1, b = 0.6, c = 0.00008, d = 2, r = 0.015 ):
    # Sets & Parameters --------------------------------
    # Define a discount factor
    rho = 1 / (1 + r) ** np.arange(T + 1)

    # State Variables -----------------------------------
    def f_f(i_S):
        return a * i_S
    
    def f_S(i_e):
        S = np.zeros(T + 1)
        S[0] = S0
        for t in range(1, T + 1):
            S[t] = S[t-1] - f_f(S[t-1]) + i_e[t-1]
            # S[t] = S[t-1] - a * S[t-1] + i_e[t-1]
        return S

    def f_W(i_e):
        S = f_S(i_e)
        W = np.zeros(T + 1)
        for t in range(T + 1):
            W[t] = rho[t] * ( (i_e[t] ** b) - (c * (S[t] ** d)) )
        return W

    # Objective Function -------------------------------
    def f_obj(i_e):
        W = f_W(i_e)
        return -1 * sum(W)


    # Constraints --------------------------------------
    cnstr = []

    # Decision Variables -------------------------------
    e_start = np.full(T+1, S0)
    bnds = [(0, 200) for t in range(T+1)]

    # Run Optimization ---------------------------------
    results = opt.minimize(f_obj, e_start, bounds=bnds, constraints=cnstr, 
                           options={'disp': False})

    # Get the optimal results
    e_opt = results.x
    S_opt = f_S(e_opt)
    W_opt = f_W(e_opt)

    # Create results dataframe
    df = pd.DataFrame({
        't': np.arange(T + 1),
        'e': e_opt,
        'f(S)': f_f(S_opt),
        'S': S_opt,
        'W': W_opt
    })
    return df

In [ ]:
# helper function for easier visualization later
def compare_stock_pollution_results(fig, df, legend_name):
    fig.add_trace(go.Scatter(x=df['t'], y=df['e'], mode='lines', name=f'e_{legend_name}', showlegend=True))
    fig.add_trace(go.Scatter(x=df['t'], y=df['f(S)'], mode='lines', name=f'f(S)_{legend_name}', showlegend=True))
    fig.add_trace(go.Scatter(x=df['t'], y=df['S'], mode='lines', name=f'S_{legend_name}', showlegend=True))
    fig.add_trace(go.Scatter(x=df['t'], y=df['W'], mode='lines', name=f'W_{legend_name}', showlegend=True))
    
    return fig

### 2a) rK-regeneration / decay function 

A local enviornmental NGO argues that given recent scientific publications, the bacteria responsible for the decay of pollutents in the lake do not reduce the stock in a linear but in a quadratic fashion. The decay rate $f(S_t)$ should be specified as illustrated below (also called an rK-function)

$$
    f(S_t) = rS_t\left(1-\frac{S_t}{K}\right) = a_1S_t\left(1-\frac{S_t}{a_2}\right)
$$

Visualize such a function for the values $r = 0.5$ and $K = 100$. What is the interpretation of these two parameters? 

### 2b) rK-function in model

Adjust your model accordingly in $f(S_t)$ and try to find an optimal emission schedule by assuming these $a_1$ and $a_2$ parameters for two scenarios. 
$$ 
a_1 =0.5 ; \qquad a_2 = 100\\
a_1 =0.8 ; \qquad a_2 = 80
$$

In [ ]:
def stockpollution_rk(T = 50,  )  # ---):
    # Sets & Parameters --------------------------------
    # Define a discount factor
    rho = 1 / (1 + r) ** np.arange(T + 1)

    # State Variables -----------------------------------
    def f_f(S_val):
        # ...

    def f_S(i_e):
        S = np.zeros(T + 1)
        S[0] = S0
        for t in range(1, T + 1):
            S[t] = S[t-1] - f_f(S[t-1]) + i_e[t-1]
        return S

    def f_W(i_e):
        S = f_S(i_e)
        W = np.zeros(T + 1)
        for t in range(T + 1):
            W[t] = rho[t] * # ...
        return W

    # Objective Function -------------------------------
    def f_obj(i_e):
        W = sum( f_W(i_e) )
        return -1 * W


    # Constraints --------------------------------------
    cnstr = []

    # Decision Variables -------------------------------
    e_start = np.full(T+1, 1.0) 
    bnds = [
        # ..
    ]

    # Run Optimization ---------------------------------
    results = opt.minimize(f_obj, e_start, bounds=bnds, constraints=cnstr, 
                           options={'disp': False})

    # Get the optimal results
    e_opt = results.x
    S_opt = f_S(e_opt)
    W_opt = f_W(e_opt)

    # Create results dataframe
    df = pd.DataFrame({
        't': np.arange(T + 1),
        'e': e_opt,
        'f(S)': f_f(S_opt),
        'S': S_opt,
        'W': W_opt
    })
    return df



In [ ]:
# df1  = ...
# df2  = ...

# Plot Results -------------------------------------
fig = go.Figure()
fig = compare_stock_pollution_results(fig, df1, 'a1=0.5, a2=100')
fig = compare_stock_pollution_results(fig, df2, 'a1=0.8, a2=80')

fig.update_layout(title='Stock Pollution', xaxis_title='time' , yaxis_title='value', template='plotly_white', width = 1000, height = 500)
fig.show()



## 3 Model in OOP-style (Object Oriented Programming)
***(For highly interested students only, not covered in class, not relevant for further course content)***

This is now for the students who are *deeply interested in coding practices* (way beyond our course objective) and have extensive prior knowledge! In recent years, it became fashionable to write your python code using classes (with attributes and methods). Try to refactor your model code for the monopolist hotelling model using object oriented programming with python classes. 

In [ ]:

class HotellingModel_Mono:
    def __init__(self, T=50, S0=100, a=10, b=1, c=1, r=0.05):
        # Initialize model parameters
        self.T = T
        self.S0 = S0
        self.a = a
        self.b = b
        self.c = c
        self.r = r
        self.rho = 1 / (1 + r) ** np.arange(T + 1)
        self.results = None
        self.df = None

    def f_S(self, i_q):
        S = np.zeros(self.T + 1)
        S[0] = self.S0
        for t in range(1, self.T + 1):
            S[t] = S[t - 1] - i_q[t - 1]
        return S

    def f_p(self, i_q):
        return self.a - self.b * i_q

    def f_obj(self, i_q):
        obj_value = sum(self.rho[t] * (i_q[t] * (self.f_p(i_q)[t] - self.c)) for t in range(self.T + 1))
        return -obj_value

    def c_total_extr(self, i_q):
        return self.S0 - sum([i_q[t] for t in range(self.T + 1)])

    def optimize(self):
        q_start = np.full(self.T + 1, 1)
        bnds = [(0, self.S0) for _ in range(self.T + 1)]
        cnstr = [{'type': 'eq', 'fun': self.c_total_extr}]

        results = opt.minimize(self.f_obj, q_start, bounds=bnds, constraints=cnstr, options={'disp': False})

        self.results = results
        q_opt = results.x
        S_opt = self.f_S(q_opt)
        p_opt = self.f_p(q_opt)
        npv_opt = self.f_obj(q_opt)

        self.df = pd.DataFrame({
            't': range(self.T + 1),
            'q_opt': q_opt,
            'S_opt': S_opt,
            'p_opt': p_opt,
            'npv_opt': npv_opt
        })

    def plot_results(self, filename='hotelling_mono.html'):
        if self.df is None:
            raise ValueError("Run the optimize method first to generate results.")
        
        fig = make_subplots(rows=1, cols=3, subplot_titles=('q_opt', 'p_opt', 'S_opt'))
        fig.add_trace(go.Scatter(x=self.df['t'], y=self.df['q_opt'], mode='lines', name='q_opt', showlegend=False), row=1, col=1)
        fig.add_trace(go.Scatter(x=self.df['t'], y=self.df['p_opt'], mode='lines', name='p_opt', showlegend=False), row=1, col=2)
        fig.add_trace(go.Scatter(x=self.df['t'], y=self.df['S_opt'], mode='lines', name='S_opt', showlegend=False), row=1, col=3)

        fig.update_layout(title='Hotelling Model', xaxis_title='t', yaxis_title='Value', template='plotly_white')
        fig.show()

    def compare_parameters(self, other_models, filename='hotelling_mono_compare.html'):
        """Compare results of different model variations"""
        fig = make_subplots(rows=1, cols=3, subplot_titles=('q_opt', 'p_opt', 'S_opt'))

        for model, label in other_models:
            model.plot_subtrace(fig, label)

        fig.update_layout(title='Hotelling Model - Parameter Comparison', xaxis_title='t', yaxis_title='Value', template='plotly_white')
        fig.write_html(filename)
        fig.show()

    def plot_subtrace(self, fig, label):
        """Helper function to add model results to a subplot"""
        fig.add_trace(go.Scatter(x=self.df['t'], y=self.df['q_opt'], mode='lines', name=f'q_{label}', legendgroup=label, showlegend=True), row=1, col=1)
        fig.add_trace(go.Scatter(x=self.df['t'], y=self.df['p_opt'], mode='lines', name=f'p_{label}', legendgroup=label, showlegend=True), row=1, col=2)
        fig.add_trace(go.Scatter(x=self.df['t'], y=self.df['S_opt'], mode='lines', name=f'S_{label}', legendgroup=label, showlegend=True), row=1, col=3)

# Example usage
if __name__ == "__main__":
    # Base model
    model_base = HotellingModel_Mono()
    model_base.optimize()
    model_base.plot_results('hotelling_mono.html')

    # Variations
    model_incr_rparam = HotellingModel_Mono(r=0.07)
    model_incr_rparam.optimize()

    model_incr_stock = HotellingModel_Mono(S0=150)
    model_incr_stock.optimize()

    model_decr_bparamm = HotellingModel_Mono(b=0.5)
    model_decr_bparamm.optimize()

    model_incr_cost = HotellingModel_Mono(c=2)
    model_incr_cost.optimize()

    # Compare models
    model_base.compare_parameters(
        [(model_incr_rparam, 'incr_rparam'), (model_incr_stock, 'incr_stock'), (model_decr_bparamm, 'decr_bparam'), (model_incr_cost, 'incr_cost')],
        filename='hotelling_mono_compare.html'
    )
